In [35]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

In [36]:
try:
    ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize()

# Spectral Change

This notebook utilizes Google Earth Engine to access Sentinel-2, Landsat 5, and Landsat 8 to view trends in spectral indices over time. The temporal trends can be year-over-year or intrayear (within one year). You can download the resulting trends as TIFFs.

You may also download the mean index value of a region for each date (image) in the image collection.

This version calculates NDVI and NDSI.

# Functions

In [37]:
def filterL_Clouds(imcol, cloud_threshold):
    ''' Filters Landsat 5 or 8 scenes based on entered threshold
        Must define cloud_threshold prior to running.
    '''
    return imcol.filter(ee.Filter.lt('CLOUD_COVER', cloud_threshold))

def maskS2clouds(image):
    """ Mask cloud and cirrus pixels from Sentinel-2 imagery using QA60 band.

        Parameters:
        image (Image): A single Image in an ImageCollection or standalone Image

        Returns:
        Image with masked features removed and original metadata

    """
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
        .And(qa.bitwiseAnd(cirrusBitMask).eq(0)) # performs bitwise AND operation between QA60 band and cloud bitmask

    return image.updateMask(mask) \
        .divide(10000) \
        .copyProperties(image, ['system:time_start']) 

def maskL_Clouds(image):
    ''' Mask Clouds and Cloud shadow pixels from Landsat 5 or 8 imagery using the QA_pixel band

        Parameters:
        image (Image): A single Image in an ImageCollection or standalone Image

        Returns:
        Image with masked features removed and original metadata

    '''
    # Bit 3 and 4 are cloud and cloud shadow, respectively
    cloud_shadow_bit_mask = (1 << 4)
    clouds_bit_mask = (1 << 3)

    qa = image.select('QA_PIXEL')
    
    # Both flags should be set to zero, indicating clear conditions
    mask = qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0).And(
           qa.bitwiseAnd(clouds_bit_mask).eq(0))

    return image.updateMask(mask)

def maskWater(image):
    ''' Mask out water using MODIS data.

        Returns: 
        Image with pixels where water_mask < 1. '''
    # Get water mask
    waterMask = (
        ee.ImageCollection('MODIS/006/MOD44W') 
        .filter(ee.Filter.date('2015-01-01', '2015-01-02')) 
        .select('water_mask') \
        .first()
    )
    
    return image.updateMask(waterMask.select('water_mask').lt(1))

def maskL_Water(image): 
    ''' Mask out water using Landsat 5 or 8 QA_pixel band.

        Returns:
        Image with masked features removed and original metadata '''
    
    # Bit 7 is water

    water_mask = (1 << 7)

    qa = image.select('QA_PIXEL')
    
    # is it zero?
    mask = qa.bitwiseAnd(water_mask).eq(0)

    return image.updateMask(mask)


def maskS2snow(image):
    ''' Mask snow from Sentinel-2 imagery with MSK_SNWPRB (snow probability mask).

        Returns: 
        Image with pixels where MSK_SNWPRB < 0.9%. '''
    
    mask = image.select('MSK_SNWPRB').lt(0.009)
    
    return image.updateMask(mask).copyProperties(image, ['system:time_start'])

def maskL_Snow(image): 
    ''' Mask snow from Landsat 5 or 8 imagery with QA_pixel band.

        Returns:
        Image with masked features removed and original metadata '''
    # Bit 5 is snow
    snow_mask = (1 << 5)

    qa = image.select('QA_PIXEL')
    
    # is it zero?
    mask = qa.bitwiseAnd(snow_mask).eq(0)

    return image.updateMask(mask)


def maskS2White(image):
    ''' Masks S2 white pixels to ensure all cloudy or snowy pixels are removed.

        Returns: 
        Image with pixels where grayscale is greater than or equal to 2000 are removed. '''

    # convert RGB values to grayscale
    grayscale = image.expression(
            '(.3 * 1e4 * R) + (.59 * 1e4 * G) + (.11 * 1e4 * B)', {
            'R': image.select('B4'),
            'G': image.select('B3'),
            'B': image.select('B2')
        })
    
    white_mask = grayscale.lte(2000)
    
    return image.updateMask(white_mask).copyProperties(image,['system:time_start'])

def calculateNoDataPercentage(image):
        """Add data on masked pixel percentage as a property

    Parameters:
    image (Image): A single Image in an ImageCollection or standalone Image

    Returns:
    Image with "nodata_percentage property" set

    """
        pixel_area = ee.Image.pixelArea()
        
        nodata_mask = (image.select('red')).eq(0) # choose any band
        nodata_area = pixel_area.updateMask(nodata_mask)
        
        nodata_area = nodata_area.reduceRegion(
              reducer=ee.Reducer.sum(),
              geometry=aoi,
              scale=30,
              maxPixels=1e9
            ).get('area')
        
        total_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=30,
            maxPixels=1e9
            ).get('area')


        nodata_pixels = ee.Number(nodata_area)
        total_pixels = ee.Number(total_area)
        
        # Calculate percentage
        nodata_percentage = nodata_pixels.divide(total_pixels).multiply(100)
        
        # Add NoData percentage as a property
        return image.set('nodata_percentage', nodata_percentage)

def clp(image):
    '''Clips a single Image to a region of interest'''
    return image.clip(aoi)

def addNDVI(image):
  '''Adds NDVI band to each image (in an ImageCollection)

    Requires redBand and NIRband be defined prior to running. '''

  ndvi = image.normalizedDifference(['NIR', 'red']).rename('NDVI')

  return image.addBands(ndvi)


In [38]:
def getS2scenes(start_year, end_year, start_month, end_month):
    ''' Build image collection with Sentinel-2 scenes and define redBand and NIRband for NDVI calculations.'''
    
    # rename S2 bands to standard names
    def _nameS2Bands(image):
        originalBands = ['B2', 'B3', 'B4', 'B8']
        namedBands = ['blue', 'green', 'red', 'NIR']
        return image.select(originalBands, namedBands)
    
    # Get Sentinel 2 harmonized images
    dataset = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filter(ee.Filter.calendarRange(start_year, end_year, 'year'))
                    .filter(ee.Filter.calendarRange(start_month, end_month, 'month'))
                    #  Pre-filter to get less cloudy granules.
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
                    .filterBounds(aoi)
                    .map(clp)
                    .map(maskS2clouds)
                    .map(maskS2snow) 
                    .map(maskS2White)
                    .map(maskWater)
                    .map(_nameS2Bands)
    )

    return dataset

def getLandsatScenes(start_year, end_year, start_month, end_month):
    # build image collection with Landsat 5 or 8, or both
    ''' Build image collection with Landsat 5, 8, or both depending on years.
    '''

    # rename S2 bands to standard names
    def _nameLandsatBands(image):

        # determine if each image is from Landsat 5 or 8
        id = image.get('SPACECRAFT_ID') 
        
        is_l5 = ee.Algorithms.IsEqual(id, 'Landsat-5') # True if Landsat-5, False if Landsat-8
        
        # select list of bands based on Landsat 5 or 8
        originalBands = ee.Algorithms.If(is_l5, 
            ['B1', 'B2', 'B3', 'B4'],  # L5
            ['B2', 'B3', 'B4', 'B5']   # L8
        )
        namedBands = ['blue', 'green', 'red', 'NIR']

        return image.select(originalBands, namedBands)

    if (start_year <= 2012 and end_year <= 2012) or (start_year >= 2013 and end_year >= 2013):

        # set bands for L5 or L8
        if end_year <= 2012:
            dataset = "LANDSAT/LT05/C02/T1_TOA"
        elif end_year >= 2013:
            dataset = "LANDSAT/LC08/C02/T1_TOA"

        # Build image collection
        cloudyScenes = (
            ee.ImageCollection(dataset) 
                        .filter(ee.Filter.calendarRange(start_year,end_year,'year'))
                        .filter(ee.Filter.calendarRange(start_month,end_month,'month'))
                        .filterBounds(aoi)
                        .map(clp)
                        )

    else:
        # merged image collection from Landsat 5 and 8 if year range spans pre and post 2012

        def _getLandsatParams(start_year,end_year):
            ''' Get Landsat 5 or 8 parameters depending on year and data availability.'''

            # empty list to contain landsat parameters:  5 or 8, and corresponding bands per sub-range of years 
            L_params = []
        
            if start_year <= 2012:
                L_params.append({
                    'dataset': 'LANDSAT/LT05/C02/T1_TOA',
                    'start_year': start_year,
                    'end_year': min(end_year, 2012),
                    'Sensor':'LT05'
                })

            if end_year >= 2013:
                L_params.append({
                    'dataset': 'LANDSAT/LC08/C02/T1_TOA',
                    'start_year': max(start_year, 2013),
                    'end_year': end_year,
                    'Sensor':'LC08'
                })
            
            return L_params


        # get landsat parameters (bands per landsat 5 or 8 depending on year)
        L_params = _getLandsatParams(start_year, end_year)

        collections = [
            ee.ImageCollection(range['dataset'])
            .filter(ee.Filter.calendarRange(range['start_year'], range['end_year'], 'year'))
            .filter(ee.Filter.calendarRange(start_month, end_month, 'month'))
            .filterBounds(aoi)
            .map(clp)
            .map(lambda img: img.set('Sensor', range['Sensor'])) # set property of each image as landsat 5 or 8
            for range in L_params
        ]

        # merge L5 and L8 collections
        cloudyScenes = ee.ImageCollection(collections[0])
        for col in collections[1:]:
            cloudyScenes = cloudyScenes.merge(col)

    dataset = (
        (filterL_Clouds(cloudyScenes, cloud_threshold))
        .map(maskL_Clouds)
        .map(maskL_Snow)
        .map(maskL_Water)
        .map(_nameLandsatBands)
        )
    
    return dataset

def annual_images(y):
    ''' Filters an image collection for a specific year (y) and date range (start and end month).
        Applies the chosen analysis (mean, min, max, or median) to calculate the statistics for the images in that year.
        
        Requires:
        index_collection, start_month, end_month, and analysis.
    
        Parameters: 
        y: a given year

        Returns:
        Statistcs over the image within the date range.
    '''
    # filter for given year
    range_year = ee.Filter.calendarRange(y, y, 'year')
    #filter for specified month range
    range_month = ee.Filter.calendarRange(start_month, end_month, 'month')

    # filter image collection by year and month and add a time band
    filtered_dataset = (index_collection
                        .filter(range_year)
                        .filter(range_month)
                        .map(lambda image: image.addBands(image.metadata('system:time_start').divide(3.154e10)))) # Needed for linear regression 
        
    # Print out the number of images in the filtered dataset
    num_images = filtered_dataset.size()
    
    # Choose the reducer based on the chosen analysis
    if analysis == 'mean':
        reducer = ee.Reducer.mean().combine( # calculates average value for each pixel across all images
            reducer2=ee.Reducer.stdDev(), # calculates standard deviation
            sharedInputs=True
        )

    elif analysis == 'min' or analysis == 'max':
        reducer = ee.Reducer.mean().combine( # calculates minimum and maximum values
            reducer2=ee.Reducer.minMax(),#
            sharedInputs=True
        )
    elif analysis == 'median':
        reducer = ee.Reducer.mean().combine( # calculates middle value for each pixel
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        )

    # Use the combined reducer to get the statistics
    stats = filtered_dataset.reduce(reducer)
    return stats.set('year', y).set('num', num_images)
    

def intrayear(index_collection):
    ''' Filter Image Collection within a single year. Used for seasonal (one-year) trends.

        Function is used when start_year = end_year.

        Requires: 
        index_collection, start_month, end_month, start_year, and end_year.

        Parameters: 
        index_collection: an ImageCollection containing images with a chosen index

        Returns:
        filtered_dataset: Filtered Image Collection
    '''
    
    range_year = ee.Filter.calendarRange(start_year, end_year, 'year')
    range_month = ee.Filter.calendarRange(start_month, end_month, 'month')
    
    # add a time band with the year to each image 
    filtered_dataset = (index_collection
                        .filter(range_year)
                        .filter(range_month)
                        .map(lambda image: image.addBands(image.metadata('system:time_start').divide(3.154e10)))) # Needed for linear regression 
    
    num_images = filtered_dataset.size()

    return filtered_dataset.set('year', start_year).set('num', num_images)

def createTimeBand(image):   
    '''Adds a time band to the image using metadata'''
    return image.addBands(image.metadata('system:time_start').divide(3.154e10))

def meanNDVI(image): 
    ''' Calculates the mean NDVI within a specific region.
        
        Requires:
        ndvi, aoi

        Returns:
        Image with property 'ndvi_mean'. '''
    
    ndvi = image.select('NDVI')
    ndvi_mean = ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10
    ).get('NDVI')

    ndvi_mean = ee.Number(ndvi_mean)
    
    return image.set('ndvi_mean', ndvi_mean)


# Build collection

### Define AOI and build image collection. Choose a cell to use HYBAS watershed boundaries, coordinates, or upload a geoJSON.

Option 1: Load the boundary of a wastershed with its level 10 HYBAS ID

In [ ]:
HYBAS_ID = 8100362730

aoi = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_10").filter(ee.Filter.eq('HYBAS_ID', HYBAS_ID))

long = aoi.geometry().centroid().coordinates().get(0).getInfo()
lat = aoi.geometry().centroid().coordinates().get(1).getInfo()

Option 2: use coordinates

In [ ]:
# lat, long  = (68.497594, -149.481197) # Toolik 

aoi_point = ee.Geometry.Point([long, lat])
aoi = aoi_point.buffer(1000).bounds() # change buffer size for different area

Option 3: upload shp from local directory

In [ ]:
aoi = geemap.shp_to_ee('.shp')    

long = aoi.geometry().centroid(maxError=1).coordinates().get(0).getInfo()
lat = aoi.geometry().centroid(maxError=1).coordinates().get(1).getInfo()

# buffer around polygon
# shp = aoi
# aoi = shp.geometry().buffer(distance=100) # meters

# Build Image Collection

Sentinel-2 imagery is only available after 2019.
If you choose Landsat, it will use L5  pre-2012 and L8 imagery post-2012. 2012 is missing data.

Enter inputs:
- satellite: 'Sentinel' or 'Landsat'
- start_year, end_year: filter image collection by months
- start_month, end_month: filter image collection by year

Enter thresholds for filtering data collection:
- cloud_threshold: percentage of cloudy pixels (0-100)
- threshold_nodata_percent: percentage of AOI that has no data (0-100)
   - *Tip: Larger AOIs and Landsat may require a larger threshold to acquire enough imagery*

In [ ]:
# Choose satellite, timespan, and filters

satellite = 'Sentinel'

# reminder that Sentinel is only available beginning in 2019
start_year, end_year = 2019, 2025
start_month, end_month = 6, 8

threshold_nodata_percent = 5
cloud_threshold = 5

In [ ]:
# build image collection
if satellite == 'Sentinel':
    im_col = getS2scenes(start_year, end_year, start_month, end_month)

elif satellite == 'Landsat':
    im_col = getLandsatScenes(start_year, end_year, start_month, end_month)

# filter for no data and add NDVI band to images
dataset = (
    im_col.map(calculateNoDataPercentage)
    .filter(ee.Filter.lte('nodata_percentage', threshold_nodata_percent))
    .map(addNDVI)
)

## Do analysis

Select your index and time frame.

- Year-over-year trend in peak greenness: NDVI max from June through August
- Year-over-year change in early season greenness: NDVI max in June (start and end month = 6)
- Seasonal, intra-year trend in greenness: NDVI max from June to August in just 2025 (start and end year = 2025)

Choosing 'max' for analysis is recommended with NDVI to avoid potential influence of snow.

In [ ]:
# Pick your index: NDVI 
index = 'NDVI'
# Choose 'mean', 'median', 'min', or 'max' for analysis
analysis = 'max'  

# image collection with one band, the index
index_collection = dataset.select(index)  

# Generate list of years
years = ee.List.sequence(start_year, end_year)

# Skip to different sections to:
- view spectral trends on the map and download
- save and plot index values

# View Trend

In [ ]:
# intrayear
if start_year == end_year:
    intrayear_collection = intrayear(index_collection)

    item = intrayear_collection.getInfo()
    print("Year:", item['properties']['year'], "Number of images:", item['properties']['num'])

    # Get linear fit to pixelwise trend of annual max NDVI
    trend = intrayear_collection.select(['system:time_start',
                                index
                                ]).reduce(ee.Reducer.linearFit())

# year-over-year
else:

    # Map over years to get yearly statistics
    yearwise_ndvi = years.map(annual_images)

    for item in yearwise_ndvi.getInfo():
        print("Year:", item['properties']['year'], "Number of images:", item['properties']['num'])

        yearCompCol = ee.ImageCollection.fromImages(yearwise_ndvi)

        # Get linear fit to pixelwise trend of annual max NDVI
        trend = yearCompCol.select(['system:time_start_mean',
                                    f'{index}_{analysis}'
                                    ]).reduce(ee.Reducer.linearFit())



### Define colorbar min and max and choose color palette.
Find the min and max values of the trend, which will be used as the min and max values of the colorbar.

In [ ]:
scale = trend.select('scale')

tMin = scale.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=aoi,
    scale=500,  # Adjust the scale depending on your resolution
    maxPixels=1e9
).getInfo()

tMax = scale.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=aoi,
    scale=500,  # Adjust the scale depending on your resolution
    maxPixels=1e9
).getInfo()

print(tMin)
print(tMax)

If the trend is only positive, make c_min = 0. 
If not, I recommend having the c_min and c_max be the same magnitude away from 0. 

Choose color palette where blue and red are positive and negative trends, respectively.

In [ ]:
# determine colorbar min and max
c_min = -0.05
c_max = 0.05

# palette
c_pal = ['red','white','blue'] # positive and negative
# c_pal = ['white','blue'] # positive

In [ ]:
# view trend
Map = geemap.Map(center=[lat, long], zoom=11)

Map.addLayer(trend.select('scale'),
              {'min': c_min, 'max': c_max,
             'palette': c_pal
             },
  'trend')

# add colorbar for trend
Map.add_colorbar_branca(colors=c_pal, vmin= c_min, vmax = c_max, layer_name='trend')

Map

download trend as TIFF

In [ ]:
geemap.download_ee_image(trend.select('scale'), 'Toolik_TGC3-6_2024.tif', scale=10, region=aoi, crs='EPSG:3995')

# Begin here to download index data

Calculate mean NDVI of the entire area for each image date in the Image Collection and convert to a data frame.

In [ ]:
# average NDVI of entire aoi per image
ndvi_collection = index_collection.map(meanNDVI)

image_list = ndvi_collection.getInfo()
data = []

# add date and mean NDVI to data frame
for img in image_list['features']:
    properties = img['properties']
    data.append({
            'date': ee.Date(properties['system:time_start']).format('YYYY-MM-dd').getInfo(),
            'ndvi_mean': properties.get('ndvi_mean')
        })
    
df = pd.DataFrame(data, columns=['date', 'ndvi_mean'])

In [ ]:
# save csv 
df.to_csv('.csv')

# Plots

load csvs

In [ ]:
index_df = pd.read_csv('.csv')

#### Convert date to datetime, add ordinal date, year, and DOY

In [ ]:
# convert date to datetime
index_df['date'] = pd.to_datetime(index_df['date'])

# add ordinal date column to assist with plotting
index_df['date_ordinal'] = index_df['date'].apply(lambda x: x.toordinal())

index_df['doy'] = index_df['date'].dt.dayofyear
index_df['year'] = index_df['date'].dt.year

plot year-over-year NDVI of an area

In [ ]:
fig = plt.figure(figsize=(9, 6))

# list of each color per year
colors = {
    2019: 'r',
    2020: 'yellow',
    2021: 'g',
    2022: 'b',
    2023: 'purple',
    2024: 'orange',
    2025: 'pink'
}

# plot data per year
for year, group in index_df.groupby(by='year'):
    plt.scatter(group['doy'], group['ndvi_mean'], label=year, color=colors[year], alpha=0.7)

# set y limits
ymin = np.nanmin(index_df['ndvi_mean'])
ymax = np.nanmax(index_df['ndvi_mean'])
plt.ylim((ymin-0.1), (ymax+0.1))

plt.xlabel('Day of Year (DOY)')
plt.ylabel('NDVI')
plt.legend()

plt.tight_layout()
plt.show()